<a href="https://colab.research.google.com/github/71percentbanana/gridathon/blob/main/Gridlock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install catboost -q
!pip install pygeohash -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.0 MB/s eta 0:00:00


In [11]:
import pandas as pd
import numpy as np
import pygeohash as pgh

from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score



df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")


for data in [df, test_df]:
    data['Temperature'] = data['Temperature'].fillna(df['Temperature'].median())
    data['RoadType'] = data['RoadType'].fillna('Unknown')
    data['Weather'] = data['Weather'].fillna('Unknown')


for data in [df, test_df]:
    data[['hour', 'minute']] = data['timestamp'].str.split(':', expand=True)

    data['hour'] = data['hour'].astype(int)
    data['minute'] = data['minute'].astype(int)

    data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
    data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)

    data['minute_sin'] = np.sin(2 * np.pi * data['minute'] / 60)
    data['minute_cos'] = np.cos(2 * np.pi * data['minute'] / 60)


for data in [df, test_df]:
    data['RoadType_Lanes'] = (
        data['RoadType'].astype(str) + "_" +
        data['NumberofLanes'].astype(str)
    )



for data in [df, test_df]:
    data['geohash_4'] = data['geohash'].str[:4]
    data['geohash_5'] = data['geohash'].str[:5]
    data['geohash_6'] = data['geohash'].str[:6]

    data['latitude'] = data['geohash'].apply(lambda x: pgh.decode(x)[0])
    data['longitude'] = data['geohash'].apply(lambda x: pgh.decode(x)[1])


features = [
    'geohash_4',
    'geohash_5',
    'geohash_6',
    'latitude',
    'longitude',
    'day',
    'RoadType',
    'NumberofLanes',
    'LargeVehicles',
    'Landmarks',
    'Temperature',
    'Weather',
    'hour_sin',
    'hour_cos',
    'minute_sin',
    'minute_cos',
    'RoadType_Lanes'
]

X = df[features]
y = df['demand']
test_X = test_df[features]

cat_features = [
    'geohash_4',
    'geohash_5',
    'geohash_6',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'RoadType_Lanes'
]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test_X))


for fold, (train_idx, val_idx) in enumerate(kf.split(X)):

    print(f"Training Fold {fold + 1}")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=2500,
        depth=9,
        learning_rate=0.03,
        loss_function='RMSE',
        random_strength=1,
        l2_leaf_reg=5,
        bagging_temperature=0.7,
        verbose=0,
        early_stopping_rounds=200
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features
    )

    val_preds = model.predict(X_val)
    oof_preds[val_idx] = val_preds

    test_preds += model.predict(test_X) / 5


rmse = np.sqrt(mean_squared_error(y, oof_preds))
r2 = r2_score(y, oof_preds)

print("\nFINAL RESULTS")
print("OOF RMSE:", rmse)
print("OOF R2:", r2)


test_preds = np.clip(test_preds, 0, 1)

submission = pd.DataFrame({
    'Index': test_df['Index'],
    'demand': test_preds
})

submission.to_csv("submission.csv", index=False)

print(submission.head())


Training with seed 42


KeyboardInterrupt: 